# Quantum Computing × Machine Learning 四種資料/演算法組合模擬（Qiskit Aer）

本筆記本會一步一步示範以下四種組合，並以同一套評估方式比較結果：

1. **Classical Data + Classical Algorithm**
2. **Classical Data + Quantum Algorithm**
3. **Quantum Data + Classical Algorithm**
4. **Quantum Data + Quantum Algorithm**

> 目標是做「教學型、可執行」的簡易模擬，重點在流程清楚與概念對照。

## 0. 安裝與匯入

如果你的環境還沒有安裝套件，可先取消下一個 cell 的註解來安裝。

In [ ]:
# 如需安裝請取消註解（在 notebook 環境執行）
# %pip install qiskit qiskit-aer scikit-learn scipy matplotlib pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

from scipy.optimize import minimize

# 固定亂數種子，讓每次結果更容易重現
RNG = np.random.default_rng(42)
np.random.seed(42)

## 1. 先準備「Classical Data」

我們使用 sklearn 的 `make_moons` 產生二元分類資料，這是常見的非線性資料集。

In [ ]:
# 產生經典（classical）資料：兩個特徵、二元標籤
X_classical, y_classical = make_moons(n_samples=300, noise=0.18, random_state=42)

# 視覺化
plt.figure(figsize=(6, 5))
plt.scatter(X_classical[:, 0], X_classical[:, 1], c=y_classical, cmap="coolwarm", s=25)
plt.title("Classical Data: make_moons")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

print("X_classical shape:", X_classical.shape)
print("y_classical shape:", y_classical.shape)

## 2. 定義「Quantum Algorithm」所需元件

這裡我們先做一個**非常簡化版**的 Variational Quantum Classifier (VQC)：

- 資料編碼（angle encoding）
- 參數化量子電路（ansatz）
- 量測第一個 qubit 的 \(\langle Z \rangle\) 作為分類分數
- 用 `scipy.optimize.minimize` 做參數訓練

> 這個版本偏向教學，重點是流程可讀，而不是追求最佳效能。

In [ ]:
def build_vqc_circuit(x, theta):
    """
    建立 2-qubit 的參數化電路
    x: 長度為 2 的輸入特徵（會做角度編碼）
    theta: 長度為 4 的可訓練參數
    """
    qc = QuantumCircuit(2)

    # --- 資料編碼（將 classical feature 映射到量子旋轉角）---
    qc.ry(x[0], 0)
    qc.ry(x[1], 1)

    # --- 可訓練 ansatz ---
    qc.rx(theta[0], 0)
    qc.ry(theta[1], 0)
    qc.rx(theta[2], 1)
    qc.ry(theta[3], 1)

    # 糾纏
    qc.cx(0, 1)

    return qc


def z_expectation_on_qubit0(statevec):
    """
    計算 qubit-0 的 <Z> 期望值。
    對 2-qubit 系統而言，基底順序是 |00>, |01>, |10>, |11>。
    qubit-0 為最右位元（Qiskit little-endian），其 Z 本徵值模式為 [+1, -1, +1, -1]。
    """
    probs = np.abs(statevec) ** 2
    z_vals = np.array([+1, -1, +1, -1])
    return float(np.dot(probs, z_vals))


def vqc_score(x, theta):
    """
    取得模型分數（0~1）。
    先取 <Z> in [-1,1]，再線性映射到 [0,1]。
    """
    qc = build_vqc_circuit(x, theta)
    state = Statevector.from_instruction(qc).data
    z = z_expectation_on_qubit0(state)
    return 0.5 * (z + 1.0)


def vqc_predict(X, theta, threshold=0.5):
    scores = np.array([vqc_score(x, theta) for x in X])
    preds = (scores >= threshold).astype(int)
    return preds, scores


def binary_cross_entropy(y_true, y_prob, eps=1e-9):
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))


def train_vqc(X_train, y_train, maxiter=80):
    """訓練簡化版 VQC，回傳最佳參數與最佳 loss。"""
    init_theta = RNG.normal(loc=0.0, scale=0.3, size=4)

    def objective(theta):
        probs = np.array([vqc_score(x, theta) for x in X_train])
        return binary_cross_entropy(y_train, probs)

    result = minimize(objective, init_theta, method="COBYLA", options={"maxiter": maxiter})
    return result.x, result.fun, result

## 3. 定義「Quantum Data」產生器

我們要讓資料本身來自量子過程：

1. 先隨機抽樣兩個角度 \(\alpha, \beta\)
2. 用小型量子電路產生量子態
3. 透過模擬器量測得到 bitstring 分佈
4. 把量測機率轉成特徵（例如 p(00), p(01), p(10), p(11)）
5. 設計一個二元標籤（例如依據某個量子可觀測量是否大於 0）

這樣得到的資料就是「quantum-born data」。

In [ ]:
sim_qasm = AerSimulator(method="automatic")


def quantum_data_circuit(alpha, beta):
    """用兩個連續參數建立量子資料來源電路。"""
    qc = QuantumCircuit(2)
    qc.ry(alpha, 0)
    qc.rx(beta, 1)
    qc.cz(0, 1)
    qc.ry(0.5 * alpha, 1)
    return qc


def measure_probabilities(alpha, beta, shots=512):
    """回傳四個基底態機率 [p00, p01, p10, p11]。"""
    qc = quantum_data_circuit(alpha, beta)
    qc_m = qc.copy()
    qc_m.measure_all()

    job = sim_qasm.run(qc_m, shots=shots)
    counts = job.result().get_counts()

    # Qiskit bitstring 以高位在左（q1 q0）顯示，這裡固定轉成 00,01,10,11 順序
    keys = ["00", "01", "10", "11"]
    probs = np.array([counts.get(k, 0) / shots for k in keys], dtype=float)
    return probs


def quantum_label_from_expectation(alpha, beta):
    """
    用 statevector 算 <Z0>，若 >=0 則標 1，否則 0。
    這裡把量子可觀測量轉成分類標籤。
    """
    state = Statevector.from_instruction(quantum_data_circuit(alpha, beta)).data
    z = z_expectation_on_qubit0(state)
    return int(z >= 0.0)


def generate_quantum_dataset(n_samples=300, shots=512, seed=42):
    rng = np.random.default_rng(seed)

    Xq = []
    yq = []

    for _ in range(n_samples):
        # alpha, beta 在 [-pi, pi] 取樣
        alpha = rng.uniform(-np.pi, np.pi)
        beta = rng.uniform(-np.pi, np.pi)

        probs = measure_probabilities(alpha, beta, shots=shots)
        label = quantum_label_from_expectation(alpha, beta)

        Xq.append(probs)
        yq.append(label)

    return np.array(Xq), np.array(yq)

## 4. 實驗 A：Classical Data + Classical Algorithm

基準組合：`make_moons` + `LogisticRegression`。

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_classical, y_classical, test_size=0.30, random_state=42, stratify=y_classical
)

# 經典演算法：標準化 + 邏輯迴歸
clf_cc = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

clf_cc.fit(X_train_c, y_train_c)
y_pred_cc = clf_cc.predict(X_test_c)
acc_cc = accuracy_score(y_test_c, y_pred_cc)

print("[A] Classical Data + Classical Algo")
print("Accuracy:", round(acc_cc, 4))
print(classification_report(y_test_c, y_pred_cc, digits=4))

## 5. 實驗 B：Classical Data + Quantum Algorithm

同一份 `make_moons` 資料，但改用簡化 VQC。

> 為了讓角度編碼穩定，我們先對特徵做標準化，再縮放到大約 \([-\pi, \pi]\) 範圍。

In [ ]:
# 先做標準化，再縮放到量子旋轉角常用範圍
scaler_for_vqc = StandardScaler()
X_train_c_scaled = scaler_for_vqc.fit_transform(X_train_c)
X_test_c_scaled = scaler_for_vqc.transform(X_test_c)

# 控制尺度，避免角度過大
X_train_c_q = np.clip(X_train_c_scaled, -2, 2) * (np.pi / 2)
X_test_c_q = np.clip(X_test_c_scaled, -2, 2) * (np.pi / 2)

theta_bc, loss_bc, result_bc = train_vqc(X_train_c_q, y_train_c, maxiter=100)
y_pred_bc, y_prob_bc = vqc_predict(X_test_c_q, theta_bc)
acc_bc = accuracy_score(y_test_c, y_pred_bc)

print("[B] Classical Data + Quantum Algo")
print("Optimized theta:", np.round(theta_bc, 4))
print("Train loss:", round(loss_bc, 4))
print("Accuracy:", round(acc_bc, 4))
print(classification_report(y_test_c, y_pred_bc, digits=4))

## 6. 實驗 C：Quantum Data + Classical Algorithm

現在改成「資料來自量子電路」，再用經典模型學習。

In [ ]:
X_quantum, y_quantum = generate_quantum_dataset(n_samples=320, shots=512, seed=123)

X_train_q, X_test_q, y_train_q, y_test_q = train_test_split(
    X_quantum, y_quantum, test_size=0.30, random_state=42, stratify=y_quantum
)

clf_qc = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

clf_qc.fit(X_train_q, y_train_q)
y_pred_qc = clf_qc.predict(X_test_q)
acc_qc = accuracy_score(y_test_q, y_pred_qc)

print("[C] Quantum Data + Classical Algo")
print("X_quantum shape:", X_quantum.shape)
print("Accuracy:", round(acc_qc, 4))
print(classification_report(y_test_q, y_pred_qc, digits=4))

## 7. 實驗 D：Quantum Data + Quantum Algorithm

最後把量子資料交給量子模型（VQC）訓練。

因為我們的 VQC 輸入是 2 維，這裡將量子資料前兩維機率特徵（p00, p01）作為示範輸入。

In [ ]:
# 取前兩個特徵做 2 維輸入，便於沿用前面的 2-qubit VQC
X_train_qq_2d = X_train_q[:, :2]
X_test_qq_2d = X_test_q[:, :2]

# 轉成角度輸入（把 [0,1] 機率映射到 [-pi, pi]）
X_train_qq_angle = (X_train_qq_2d * 2 - 1) * np.pi
X_test_qq_angle = (X_test_qq_2d * 2 - 1) * np.pi

theta_dq, loss_dq, result_dq = train_vqc(X_train_qq_angle, y_train_q, maxiter=100)
y_pred_dq, y_prob_dq = vqc_predict(X_test_qq_angle, theta_dq)
acc_dq = accuracy_score(y_test_q, y_pred_dq)

print("[D] Quantum Data + Quantum Algo")
print("Optimized theta:", np.round(theta_dq, 4))
print("Train loss:", round(loss_dq, 4))
print("Accuracy:", round(acc_dq, 4))
print(classification_report(y_test_q, y_pred_dq, digits=4))

## 8. 結果總結比較

我們把四種搭配的 accuracy 放在同一張表方便比較。

In [ ]:
summary_df = pd.DataFrame([
    {"Setting": "A: Classical Data + Classical Algo", "Accuracy": acc_cc},
    {"Setting": "B: Classical Data + Quantum Algo", "Accuracy": acc_bc},
    {"Setting": "C: Quantum Data + Classical Algo", "Accuracy": acc_qc},
    {"Setting": "D: Quantum Data + Quantum Algo", "Accuracy": acc_dq},
]).sort_values("Accuracy", ascending=False)

summary_df

## 9. 重點解讀（中文）

1. **A 組（經典+經典）**通常是穩定基準，訓練速度也最快。
2. **B 組（經典+量子）**展示了「把經典資料編碼進量子電路」的流程，效果受 ansatz 與優化器影響很大。
3. **C 組（量子+經典）**常是務實做法：資料由量子系統生成，但建模用成熟經典方法。
4. **D 組（量子+量子）**概念最完整，但在小規模模擬下未必優於經典；其價值在於探索量子特徵結構。

---

### 你可以繼續延伸

- 把 VQC 換成更深的 ansatz（更多參數、更多糾纏層）
- 改用 `Estimator`/`Sampler` primitive 工作流
- 用 kernel 方法（Quantum Kernel + SVM）
- 比較不同 shots 對 quantum data 噪聲的影響